# 02 — Several retrieval mixtures at each position

A single head uses one source distribution for all its value coordinates. Multiple heads permit several distributions; they do not split the sentence into separate token ranges.

For $X:[B,T,D]$, each projected tensor is reshaped to $[B,H,T,d]$. Each head computes $O_h=\mathrm{softmax}(Q_hK_h^\top/\sqrt d+M)V_h$. Concatenate on the feature axis, then apply $W_O$. This output projection is not the vocabulary head.

## How to work through this notebook

Run setup once. At each checkpoint, write a prediction and try the small implementation before reading its adjacent reference solution. All reference cells run unchanged from top to bottom; exercise cells contain safe, optional starting points. Numerical checks use CPU float64 unless explicitly noted. Agent-verified reference execution is separate from your learning progress.

In [ ]:
from pathlib import Path
import sys, copy, math, inspect
from dataclasses import replace
import torch
from torch import nn
from torch.nn import functional as F
root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src/dongxi_llms/decoder_lab.py").exists()), None)
if root is None:
    raise RuntimeError("Open this notebook from inside the Dongxi_LLMs repository")
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
from dongxi_llms.decoder_lab import (
    DecoderConfig, TinyDecoder, DecoderBlock, MultiHeadAttention, MLP, RMSNorm,
    layer_norm, rms_norm, rope, attend, parameter_count, analytical_parameters,
    cost_estimate, teaching_batch, next_token_loss, fit_one_batch)
torch.set_num_threads(1)
torch.manual_seed(505)
DTYPE = torch.float64
def close(actual, expected, atol=1e-10, rtol=1e-8):
    torch.testing.assert_close(actual, expected, atol=atol, rtol=rtol)
print("CPU reference environment:", torch.__version__)


## 1. Implement the head split

Project one shared input into Q/K/V, then split heads. Which axis represents time, and which represents a retrieval head?

**Your prediction:** _Write it here before running the reference._

In [ ]:
cfg = DecoderConfig()
mha = MultiHeadAttention(cfg).double()
x = torch.randn(2, 6, 16, dtype=DTYPE, requires_grad=True)
# Your implementation: q = ...; k = ...; v = ...

### Reference solution

Run after your attempt; compare the mechanism, not just the final numbers.

In [ ]:
def split(proj):
    return proj(x).reshape(2, 6, 4, 4).transpose(1, 2)
q, k, v = split(mha.q), split(mha.k), split(mha.v)
print("X:", x.shape, "Q/K/V:", q.shape, k.shape, v.shape)
print(inspect.getsource(MultiHeadAttention.forward))

### Why this works

The transpose changes [batch,time,head,feature] to [batch,head,time,feature]. It must be reversed before head features are flattened. Merely reshaping a transposed tensor to the desired output size can silently mix axes.

## 2. Compare independent heads with vectorized attention

Compute each head in a loop, then compare outputs AND input gradients with the vectorized module using identical weights.

**Your prediction:** _Write it here before running the reference._

In [ ]:
# Your implementation: per_head = [...]; merged = ...

### Reference solution

Run after your attempt; compare the mechanism, not just the final numbers.

In [ ]:
per_head = [attend(q[:, h:h+1], k[:, h:h+1], v[:, h:h+1])[0]
            for h in range(cfg.heads)]
mixed = torch.cat(per_head, dim=1)
merged = mixed.transpose(1, 2).contiguous().reshape(2, 6, 16)
manual = mha.out(merged)
actual, _, weights = mha(x)
close(manual, actual)
probe = torch.randn_like(actual)
ga = torch.autograd.grad((manual*probe).sum(), x, retain_graph=True)[0]
gb = torch.autograd.grad((actual*probe).sum(), x, retain_graph=True)[0]
close(ga, gb)
print("Output error:", float((manual-actual).abs().max().detach()))
print("Gradient error:", float((ga-gb).abs().max()))
print("Head source distributions at the last position:", weights[0, :, -1].detach())

### Why this works

Both forms implement the same operation. Vectorization changes the computation layout, not the learning rule or the tokens a head may access.

## 3. Ablate a head and perturb the future

Zero one head's retrieved content without changing its attention probabilities. Separately change future inputs. Which outputs are allowed to change?

**Your prediction:** _Write it here before running the reference._

In [ ]:
# Predict both interventions before running the reference.

### Reference solution

Run after your attempt; compare the mechanism, not just the final numbers.

In [ ]:
ablated = mixed.clone()
ablated[:, 0] = 0
ablated_output = mha.out(ablated.transpose(1, 2).contiguous().reshape(2, 6, 16))
assert not torch.allclose(ablated_output, actual)
changed = x.detach().clone(); changed[:, 4:] += 3
changed_output = mha(changed)[0]
close(changed_output[:, :4], actual[:, :4])
assert weights.triu(1).abs().max() == 0
print("Head-zeroing output change:", float((ablated_output-actual).norm().detach()))
print("Earlier-position future perturbation error:",
      float((changed_output[:, :4]-actual[:, :4]).abs().max().detach()))

### Why this works

Zeroing a value mixture removes its contribution before W_O. Causal masking separately protects earlier outputs. Neither an attention map nor this random head ablation establishes a fixed human-readable head role.

## Takeaway and evidence boundary

Next: attention has produced an update of width D. The residual connection determines how this update joins the existing state.

Companion map: [Chapter 5 pathway](../day-05/README.md). Reusable source: [decoder_lab.py](../../src/dongxi_llms/decoder_lab.py). Record your explanation and remaining questions here; the notebook's existence does not mark the lesson complete.